# 08. Attention core — MLA and DeepSeek-V4 CSA/HCA

Only sequence length, width and head counts are reduced. The disclosed V4 attention paths are kept: sequence compression, Lightning-style indexing, local sliding-window context, compressed sparse context, partial RoPE, attention sink, HCA, and grouped low-rank output projection.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. MHA / MQA / GQA cache geometry


In [ ]:
batch = 2
length = 16
head_dim = 8
query_heads = 4

mha_k = torch.randn(batch, query_heads, length, head_dim, device=device)
mqa_k = torch.randn(batch, 1, length, head_dim, device=device)
gqa_k = torch.randn(batch, 2, length, head_dim, device=device)

print("MHA K elements:", mha_k.numel())
print("MQA K elements:", mqa_k.numel())
print("GQA K elements:", gqa_k.numel())


## 2. RoPE on only the positional subspace


In [ ]:
def apply_rope(x, positions):
    dim = x.size(-1)
    assert dim % 2 == 0

    index = torch.arange(
        0,
        dim,
        2,
        device=x.device,
        dtype=torch.float32,
    )
    inverse_frequency = 1.0 / (10000 ** (index / dim))
    angles = positions.float()[:, None] * inverse_frequency[None]

    while angles.dim() < x.dim():
        angles = angles.unsqueeze(0)

    even = x[..., 0::2]
    odd = x[..., 1::2]
    rotated_even = even * angles.cos() - odd * angles.sin()
    rotated_odd = even * angles.sin() + odd * angles.cos()
    return torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    ).flatten(-2)


## 3. DeepSeek-V2 style MLA baseline


In [ ]:
class TinyMLA(nn.Module):
    def __init__(
        self,
        model_dim=32,
        heads=4,
        q_rank=12,
        kv_rank=8,
        content_dim=6,
        rope_dim=2,
        value_dim=6,
    ):
        super().__init__()
        self.heads = heads
        self.kv_rank = kv_rank
        self.content_dim = content_dim
        self.rope_dim = rope_dim
        self.value_dim = value_dim

        self.q_down = nn.Linear(model_dim, q_rank, bias=False)
        self.q_norm = nn.RMSNorm(q_rank)
        self.q_up = nn.Linear(
            q_rank,
            heads * (content_dim + rope_dim),
            bias=False,
        )
        self.kv_down = nn.Linear(
            model_dim,
            kv_rank + rope_dim,
            bias=False,
        )
        self.kv_norm = nn.RMSNorm(kv_rank)
        self.kv_up = nn.Linear(
            kv_rank,
            heads * (content_dim + value_dim),
            bias=False,
        )
        self.out = nn.Linear(heads * value_dim, model_dim, bias=False)

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        positions = torch.arange(length, device=hidden.device)

        q_latent = self.q_norm(self.q_down(hidden))
        q = self.q_up(q_latent).view(
            batch,
            length,
            self.heads,
            self.content_dim + self.rope_dim,
        ).transpose(1, 2)
        q_content, q_rope = q.split(
            [self.content_dim, self.rope_dim],
            dim=-1,
        )

        compressed = self.kv_down(hidden)
        kv_latent, shared_rope = compressed.split(
            [self.kv_rank, self.rope_dim],
            dim=-1,
        )
        kv = self.kv_up(self.kv_norm(kv_latent)).view(
            batch,
            length,
            self.heads,
            self.content_dim + self.value_dim,
        ).transpose(1, 2)
        k_content, value = kv.split(
            [self.content_dim, self.value_dim],
            dim=-1,
        )
        k_rope = shared_rope[:, None].expand(-1, self.heads, -1, -1)

        query = torch.cat([q_content, apply_rope(q_rope, positions)], dim=-1)
        key = torch.cat([k_content, apply_rope(k_rope, positions)], dim=-1)
        attended = F.scaled_dot_product_attention(
            query,
            key,
            value,
            is_causal=True,
        )
        attended = attended.transpose(1, 2).contiguous().flatten(2)
        return self.out(attended), {
            "kv_latent": kv_latent,
            "shared_rope": shared_rope,
        }


hidden = torch.randn(2, 12, 32, device=device)
mla = TinyMLA().to(device)
mla_output, mla_diag = mla(hidden)
print("MLA output:", mla_output.shape)
print("latent cache:", mla_diag["kv_latent"].shape)


## 4. Overlapping sequence compression


In [ ]:
class SequenceCompressor(nn.Module):
    def __init__(
        self,
        model_dim=32,
        compressed_dim=16,
        window=4,
        stride=2,
    ):
        super().__init__()
        self.window = window
        self.stride = stride
        self.score = nn.Linear(model_dim, 1)
        self.key = nn.Linear(model_dim, compressed_dim, bias=False)
        self.value = nn.Linear(model_dim, compressed_dim, bias=False)

    def forward(self, hidden):
        windows = hidden.unfold(1, self.window, self.stride)
        windows = windows.permute(0, 1, 3, 2).contiguous()
        weights = self.score(windows).squeeze(-1).softmax(dim=-1)
        pooled = torch.einsum("bnw,bnwd->bnd", weights, windows)

        count = pooled.size(1)
        positions = (
            torch.arange(count, device=hidden.device) * self.stride
            + self.window - 1
        )
        positions = positions.clamp_max(hidden.size(1) - 1)
        return self.key(pooled), self.value(pooled), pooled, positions


## 5. Grouped low-rank output projection


In [ ]:
class GroupedOutputProjection(nn.Module):
    def __init__(self, input_dim=32, output_dim=32, groups=2, rank=8):
        super().__init__()
        assert input_dim % groups == 0
        self.groups = groups
        self.group_dim = input_dim // groups
        self.down = nn.ModuleList(
            [nn.Linear(self.group_dim, rank, bias=False) for _ in range(groups)]
        )
        self.up = nn.Linear(groups * rank, output_dim, bias=False)

    def forward(self, x):
        chunks = x.split(self.group_dim, dim=-1)
        compressed = [projection(chunk) for projection, chunk in zip(self.down, chunks)]
        return self.up(torch.cat(compressed, dim=-1))


## 6. Full small CSA: Lightning indexer + SWA + compressed top-k + sink

For each query token, the expensive attention is evaluated only against its recent sliding-window tokens and the compressed entries selected by the learned indexer. The sink is represented as an extra learnable logit with a zero value, so it participates in the softmax denominator without inventing a content vector.


In [ ]:
class FullCompressedSparseAttention(nn.Module):
    def __init__(
        self,
        model_dim=32,
        heads=4,
        head_dim=8,
        rope_dim=2,
        top_k=3,
        local_window=4,
    ):
        super().__init__()
        self.heads = heads
        self.head_dim = head_dim
        self.rope_dim = rope_dim
        self.content_dim = head_dim - rope_dim
        self.top_k = top_k
        self.local_window = local_window
        inner = heads * head_dim

        self.query = nn.Linear(model_dim, inner, bias=False)
        self.local_key = nn.Linear(model_dim, inner, bias=False)
        self.local_value = nn.Linear(model_dim, inner, bias=False)
        self.compressor = SequenceCompressor(
            model_dim,
            compressed_dim=inner,
            window=4,
            stride=2,
        )
        self.index_query = nn.Linear(model_dim, 16, bias=False)
        self.index_key = nn.Linear(model_dim, 16, bias=False)
        self.sink_logit = nn.Parameter(torch.zeros(heads))
        self.output = GroupedOutputProjection(inner, model_dim, groups=2, rank=8)

    def split_heads(self, x):
        return x.view(x.size(0), x.size(1), self.heads, self.head_dim)

    def apply_partial_rope(self, x, positions):
        content = x[..., : self.content_dim]
        positional = x[..., self.content_dim :]
        positional = positional.permute(0, 2, 1, 3)
        positional = apply_rope(positional, positions)
        positional = positional.permute(0, 2, 1, 3)
        return torch.cat([content, positional], dim=-1)

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        positions = torch.arange(length, device=hidden.device)

        query = self.apply_partial_rope(
            self.split_heads(self.query(hidden)),
            positions,
        )
        local_key = self.apply_partial_rope(
            self.split_heads(self.local_key(hidden)),
            positions,
        )
        local_value = self.split_heads(self.local_value(hidden))

        compressed_key, compressed_value, summary, compressed_positions = (
            self.compressor(hidden)
        )
        compressed_key = self.apply_partial_rope(
            self.split_heads(compressed_key),
            compressed_positions,
        )
        compressed_value = self.split_heads(compressed_value)

        index_q = F.normalize(self.index_query(hidden), dim=-1)
        index_k = F.normalize(self.index_key(summary), dim=-1)
        index_scores = torch.einsum(
            "btd,bnd->btn",
            index_q,
            index_k,
        )
        selected_ids = index_scores.topk(
            min(self.top_k, summary.size(1)),
            dim=-1,
        ).indices

        outputs = []
        fine_score_count = 0

        for token_index in range(length):
            local_start = max(0, token_index - self.local_window + 1)
            local_k = local_key[:, local_start : token_index + 1]
            local_v = local_value[:, local_start : token_index + 1]

            chosen = selected_ids[:, token_index]
            batch_ids = torch.arange(batch, device=hidden.device)[:, None]
            sparse_k = compressed_key[batch_ids, chosen]
            sparse_v = compressed_value[batch_ids, chosen]

            candidate_k = torch.cat([local_k, sparse_k], dim=1)
            candidate_v = torch.cat([local_v, sparse_v], dim=1)

            q_t = query[:, token_index]
            scores = torch.einsum("bhd,bkhd->bhk", q_t, candidate_k)
            scores = scores / math.sqrt(self.head_dim)

            sink = self.sink_logit[None, :, None].expand(batch, -1, 1)
            scores_with_sink = torch.cat([scores, sink], dim=-1)
            weights = scores_with_sink.softmax(dim=-1)[..., :-1]
            attended = torch.einsum("bhk,bkhd->bhd", weights, candidate_v)
            outputs.append(attended)
            fine_score_count += scores.numel()

        output = torch.stack(outputs, dim=1).flatten(2)
        return self.output(output), {
            "selected_ids": selected_ids,
            "fine_score_count": fine_score_count,
        }


csa = FullCompressedSparseAttention().to(device)
csa_output, csa_diag = csa(hidden)
print("CSA output:", csa_output.shape)
print("selected compressed entries:", csa_diag["selected_ids"].shape)
print("fine-score count:", csa_diag["fine_score_count"])


## 7. HCA: heavier compression followed by dense attention


In [ ]:
class HeavilyCompressedAttention(nn.Module):
    def __init__(self, model_dim=32, heads=4, head_dim=8):
        super().__init__()
        self.heads = heads
        self.head_dim = head_dim
        inner = heads * head_dim
        self.query = nn.Linear(model_dim, inner, bias=False)
        self.compressor = SequenceCompressor(
            model_dim,
            compressed_dim=inner,
            window=6,
            stride=4,
        )
        self.sink_logit = nn.Parameter(torch.zeros(heads))
        self.output = GroupedOutputProjection(inner, model_dim, groups=2, rank=8)

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        key, value, _, _ = self.compressor(hidden)
        query = self.query(hidden).view(batch, length, self.heads, self.head_dim)
        key = key.view(batch, -1, self.heads, self.head_dim)
        value = value.view(batch, -1, self.heads, self.head_dim)

        scores = torch.einsum("bthd,bnhd->bhtn", query, key)
        scores = scores / math.sqrt(self.head_dim)
        sink = self.sink_logit[None, :, None, None].expand(batch, -1, length, 1)
        weights = torch.cat([scores, sink], dim=-1).softmax(dim=-1)[..., :-1]
        attended = torch.einsum("bhtn,bnhd->bthd", weights, value).flatten(2)
        return self.output(attended)


hca = HeavilyCompressedAttention().to(device)
hca_output = hca(hidden)
print("HCA output:", hca_output.shape)


## 8. Hybrid V4 attention block


In [ ]:
class V4HybridAttentionBlock(nn.Module):
    def __init__(self, model_dim=32):
        super().__init__()
        self.norm_csa = nn.RMSNorm(model_dim)
        self.norm_hca = nn.RMSNorm(model_dim)
        self.csa = FullCompressedSparseAttention(model_dim=model_dim)
        self.hca = HeavilyCompressedAttention(model_dim=model_dim)
        self.mix = nn.Linear(2 * model_dim, model_dim, bias=False)

    def forward(self, hidden):
        csa_output, diagnostics = self.csa(self.norm_csa(hidden))
        hca_output = self.hca(self.norm_hca(hidden))
        update = self.mix(torch.cat([csa_output, hca_output], dim=-1))
        return hidden + update, diagnostics


v4 = V4HybridAttentionBlock().to(device)
v4_output, diagnostics = v4(hidden)
loss = v4_output.square().mean()
loss.backward()
print("hybrid output:", v4_output.shape)
print("indexer grad:", v4.csa.index_query.weight.grad.norm().item())
print("sink grad:", v4.csa.sink_logit.grad.norm().item())


## References and provenance

- MLA: low-rank latent KV cache with a decoupled positional subspace.
- DeepSeek-V4 public implementation/docs: sequence-axis KV compression, Lightning Indexer top-k compressed selection, recent sliding-window KV, partial RoPE, learnable attention sink, grouped low-rank output projection, and a more heavily compressed dense-attention path.
- Production kernels and billion-parameter dimensions are reduced; these disclosed computation paths are not replaced by generic attention.
